# Day 32 AM — Decision Trees & Random Forest
## Bank Loan Approval System
**Week 6 | Machine Learning & AI**

---
**Topics:** Gini impurity, entropy, information gain, overfitting & pruning, Decision Tree hyperparameters,  
bagging & bootstrap, feature randomness, Random Forest hyperparameters, feature importance, GridSearchCV, RandomizedSearchCV, cross-validation

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import (train_test_split, RandomizedSearchCV,
                                      cross_val_score, StratifiedKFold)
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from scipy.stats import randint, uniform

SEED = 42
np.random.seed(SEED)
print('All libraries loaded ✓')

---
## Part A — Concept Application (40%)
### A1 · Create Synthetic Loan Data (2 000 records)

In [ ]:
def generate_loan_data(n=2000, seed=SEED):
    """Generate realistic synthetic loan approval data."""
    rng = np.random.default_rng(seed)

    annual_income      = rng.normal(65_000, 25_000, n).clip(20_000, 250_000)
    credit_score       = rng.normal(680, 75, n).clip(300, 850).astype(int)
    loan_amount        = rng.normal(20_000, 12_000, n).clip(1_000, 100_000)
    employment_years   = rng.exponential(5, n).clip(0, 35)
    debt_to_income     = rng.beta(2, 5, n)            # mostly low, right-skewed
    num_credit_cards   = rng.integers(0, 12, n)

    # Business rule-based approval (ground truth)
    score = (
        (credit_score > 700).astype(float) * 3.0
        + (debt_to_income < 0.35).astype(float) * 2.0
        + (annual_income > 50_000).astype(float) * 1.5
        + (employment_years > 3).astype(float) * 1.0
        - (loan_amount / annual_income > 0.5).astype(float) * 2.0
    )
    noise = rng.normal(0, 0.5, n)
    approved = (score + noise > 3.5).astype(int)

    return pd.DataFrame({
        'annual_income'    : annual_income.round(2),
        'credit_score'     : credit_score,
        'loan_amount'      : loan_amount.round(2),
        'employment_years' : employment_years.round(2),
        'debt_to_income'   : debt_to_income.round(4),
        'num_credit_cards' : num_credit_cards,
        'approved'         : approved
    })


df = generate_loan_data()
print(f'Shape: {df.shape}  |  Approval rate: {df.approved.mean():.1%}')
df.describe().round(2)

### A2 · Decision Tree (max_depth=4) + Top 3 Decision Rules

In [ ]:
FEATURES = ['annual_income','credit_score','loan_amount',
            'employment_years','debt_to_income','num_credit_cards']

X = df[FEATURES]
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

# ── Train Decision Tree ───────────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=4, random_state=SEED)
dt.fit(X_train, y_train)

# Visualise
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(dt, feature_names=FEATURES, class_names=['REJECT','APPROVE'],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree (max_depth=4) — Loan Approval', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dt_tree.png', dpi=120, bbox_inches='tight')
plt.show()
print('Tree saved as dt_tree.png')

In [ ]:
# ── Extract Decision Rules ────────────────────────────────────────────────────
def extract_top_rules(tree, feature_names, top_n=3):
    """Walk tree paths to leaf nodes; return top-N by sample support."""
    tree_ = tree.tree_
    feat  = [feature_names[i] if i != -2 else 'leaf'
             for i in tree_.feature]
    rules = []

    def recurse(node, conditions):
        if tree_.feature[node] == -2:           # leaf
            n_samples = int(tree_.n_node_samples[node])
            values    = tree_.value[node][0]
            n_class   = int(np.argmax(values))
            purity    = values[n_class] / values.sum()
            label     = 'APPROVE' if n_class == 1 else 'REJECT'
            rules.append((n_samples, purity, label, list(conditions)))
        else:
            thresh = round(tree_.threshold[node], 2)
            f      = feat[node]
            recurse(tree_.children_left[node],
                    conditions + [f'{f} <= {thresh}'])
            recurse(tree_.children_right[node],
                    conditions + [f'{f} > {thresh}'])

    recurse(0, [])
    rules.sort(key=lambda x: x[0], reverse=True)
    return rules[:top_n]


top_rules = extract_top_rules(dt, FEATURES)

print('=' * 65)
print('TOP 3 DECISION RULES  (sorted by sample support)')
print('=' * 65)
for i, (n_samples, purity, label, conds) in enumerate(top_rules, 1):
    cond_str = ' AND '.join(conds)
    print(f'\nRule {i}: IF {cond_str}')
    print(f'        → {label}  (purity {purity:.0%}, n={n_samples})')

### A3 · Random Forest — Tuned with RandomizedSearchCV (5-fold CV)

In [ ]:
param_dist = {
    'n_estimators'      : randint(100, 500),
    'max_depth'         : [None, 5, 10, 15, 20],
    'min_samples_split' : randint(2, 20),
    'min_samples_leaf'  : randint(1, 10),
    'max_features'      : ['sqrt', 'log2', 0.5, 0.7],
    'class_weight'      : [None, 'balanced'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=40,
    scoring='roc_auc',
    cv=cv,
    random_state=SEED,
    verbose=0,
    refit=True,
)

rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_

print('Best params:', rf_search.best_params_)
print(f'Best CV ROC-AUC: {rf_search.best_score_:.4f}')

### A4 · Compare Models: Accuracy, F1, ROC-AUC, Interpretability

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, name):
    """Return dict of key metrics for a fitted model."""
    y_pred   = model.predict(X_te)
    y_prob   = model.predict_proba(X_te)[:, 1]
    cv_scores= cross_val_score(model, X_tr, y_tr, cv=5,
                               scoring='roc_auc', n_jobs=-1)
    return {
        'Model'         : name,
        'Accuracy'      : accuracy_score(y_te, y_pred),
        'F1'            : f1_score(y_te, y_pred),
        'ROC-AUC'       : roc_auc_score(y_te, y_prob),
        'CV ROC-AUC μ'  : cv_scores.mean(),
        'CV ROC-AUC σ'  : cv_scores.std(),
    }


results = pd.DataFrame([
    evaluate_model(dt,      X_train, y_train, X_test, y_test, 'Decision Tree (d=4)'),
    evaluate_model(rf_best, X_train, y_train, X_test, y_test, 'Random Forest (tuned)'),
])
results.set_index('Model', inplace=True)
print(results.round(4).to_string())

# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, model, title in zip(axes,
                             [dt, rf_best],
                             ['Decision Tree (d=4)', 'Random Forest (tuned)']):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=['REJECT','APPROVE'],
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)
plt.suptitle('Confusion Matrices — Loan Approval', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

### A5 · Permutation Importance vs Default Feature Importances

In [ ]:
# MDI (default feature_importances_)
mdi_imp = pd.Series(rf_best.feature_importances_, index=FEATURES, name='MDI')

# Permutation importance on test set
perm_result = permutation_importance(
    rf_best, X_test, y_test,
    n_repeats=20, random_state=SEED, scoring='roc_auc', n_jobs=-1)
perm_imp = pd.Series(perm_result.importances_mean, index=FEATURES, name='Permutation')

imp_df = pd.concat([mdi_imp, perm_imp], axis=1).sort_values('Permutation', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2196F3', '#FF9800']

for ax, col, color in zip(axes, ['MDI', 'Permutation'], colors):
    imp_df[col].sort_values().plot.barh(ax=ax, color=color, edgecolor='white')
    ax.set_title(f'{col} Importance', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance score')
    ax.axvline(0, color='black', linewidth=0.8)

plt.suptitle('MDI vs Permutation Feature Importance — Random Forest',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nRanking comparison:')
print(imp_df.sort_values('Permutation', ascending=False).round(4).to_string())

### A6 · Deployment Recommendation (1 paragraph)

In [ ]:
recommendation = """
DEPLOYMENT RECOMMENDATION
==========================
The tuned Random Forest outperforms the Decision Tree on every quantitative
metric (higher accuracy, F1, and ROC-AUC) while exhibiting much lower
variance across cross-validation folds, making its real-world performance
more predictable. However, banking regulators require model explanations
for credit decisions under regulations such as the Equal Credit Opportunity
Act (ECOA). We therefore recommend a HYBRID approach: deploy the Random
Forest as the scoring engine (automated decisions), and use the extracted
Decision Tree rules as human-readable explanations presented to applicants
and auditors whenever a loan is declined. The three key rules — centred on
credit_score > 700, debt_to_income < 0.35, and employment_years — are
legally defensible, intuitive, and consistent with standard underwriting
practice. This dual-model architecture satisfies both the accuracy
requirement (RF) and the interpretability mandate (DT rules).
"""
print(recommendation)

---
## Part B — Stretch Problem (30%): ExtraTreesClassifier vs Random Forest

In [ ]:
import time

def benchmark(model, X_tr, y_tr, X_te, y_te, n_runs=5):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.fit(X_tr, y_tr)
        times.append(time.perf_counter() - t0)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    return {
        'accuracy' : accuracy_score(y_te, y_pred),
        'roc_auc'  : roc_auc_score(y_te, y_prob),
        'train_sec': np.mean(times),
    }


N_EST = 200

rf_bench  = RandomForestClassifier(n_estimators=N_EST, random_state=SEED, n_jobs=-1)
et_bench  = ExtraTreesClassifier  (n_estimators=N_EST, random_state=SEED, n_jobs=-1)

rf_stats  = benchmark(rf_bench, X_train, y_train, X_test, y_test)
et_stats  = benchmark(et_bench, X_train, y_train, X_test, y_test)

comparison = pd.DataFrame([rf_stats, et_stats],
                          index=['Random Forest', 'Extra Trees'])
print(comparison.round(4).to_string())
print(f'\nSpeed-up (ET / RF): {rf_stats["train_sec"] / et_stats["train_sec"]:.2f}x faster')

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = ['accuracy', 'roc_auc', 'train_sec']
labels  = ['Accuracy', 'ROC-AUC', 'Train Time (s)']
colors  = ['#1E88E5', '#43A047']

for ax, m, lbl in zip(axes, metrics, labels):
    vals = [rf_stats[m], et_stats[m]]
    bars = ax.bar(['Random Forest', 'Extra Trees'], vals, color=colors, width=0.5, edgecolor='white')
    ax.set_title(lbl, fontsize=12)
    ax.set_ylim(0, max(vals) * 1.2)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.02,
                f'{v:.4f}', ha='center', fontsize=10)

plt.suptitle('Random Forest vs Extra Trees — Loan Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_vs_et.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Part C — Interview Ready (20%)

### Q1 · Bias-Variance Tradeoff: DT (high variance) vs RF (lower variance)

In [ ]:
# Simulate bias-variance via bootstrap trials
N_TRIALS = 50
dt_preds, rf_preds = [], []

for i in range(N_TRIALS):
    idx = np.random.choice(len(X_train), len(X_train), replace=True)
    X_b, y_b = X_train.iloc[idx], y_train.iloc[idx]

    dt_t = DecisionTreeClassifier(random_state=i).fit(X_b, y_b)
    rf_t = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=i).fit(X_b, y_b)

    dt_preds.append(dt_t.predict(X_test))
    rf_preds.append(rf_t.predict(X_test))

dt_arr = np.array(dt_preds).T    # shape (n_test, n_trials)
rf_arr = np.array(rf_preds).T

dt_variance = dt_arr.var(axis=1).mean()
rf_variance = rf_arr.var(axis=1).mean()

print(f'Mean prediction variance — Decision Tree : {dt_variance:.4f}')
print(f'Mean prediction variance — Random Forest : {rf_variance:.4f}')
print(f'Variance reduction by RF: {(1 - rf_variance/dt_variance):.1%}')

In [ ]:
# Bias-Variance diagram
depths     = range(1, 20)
dt_train   = []
dt_test    = []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED).fit(X_train, y_train)
    dt_train.append(accuracy_score(y_train, m.predict(X_train)))
    dt_test.append(accuracy_score(y_test,  m.predict(X_test)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overfitting curve
ax = axes[0]
ax.plot(depths, dt_train, 'b-o', label='Train accuracy', markersize=5)
ax.plot(depths, dt_test,  'r-o', label='Test accuracy',  markersize=5)
opt_depth = depths[int(np.argmax(dt_test))]
ax.axvline(opt_depth, ls='--', color='green', label=f'Optimal depth={opt_depth}')
ax.fill_between(depths, dt_train, dt_test, alpha=0.12, color='purple',
                label='Variance gap')
ax.set_xlabel('max_depth'); ax.set_ylabel('Accuracy')
ax.set_title('DT Overfitting Curve', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Right: conceptual bias-variance diagram
ax2 = axes[1]
x_axis = np.linspace(0, 10, 200)
bias_sq  = 4 * np.exp(-0.4 * x_axis)          # decreasing
variance = 0.3 * np.exp(0.35 * x_axis)        # increasing
total    = bias_sq + variance + 0.5            # irreducible error

ax2.plot(x_axis, bias_sq,  label='Bias²',            color='#E53935', lw=2)
ax2.plot(x_axis, variance, label='Variance',          color='#1E88E5', lw=2)
ax2.plot(x_axis, total,    label='Total Error',       color='#43A047', lw=2, ls='--')
ax2.axhline(0.5, ls=':', color='gray', label='Irreducible error')

opt_x = x_axis[np.argmin(total)]
ax2.axvline(opt_x, ls='--', color='black', lw=1.5, label=f'Sweet spot')

ax2.annotate('High bias\n(DT underfits)', xy=(0.5, bias_sq[10]),
             xytext=(1.5, 3.5), fontsize=8, color='#E53935',
             arrowprops=dict(arrowstyle='->', color='#E53935'))
ax2.annotate('High variance\n(DT overfits)', xy=(9, variance[190]),
             xytext=(6, 3.5), fontsize=8, color='#1E88E5',
             arrowprops=dict(arrowstyle='->', color='#1E88E5'))
ax2.annotate('RF via bagging\nreduces variance', xy=(opt_x, total[np.argmin(total)]),
             xytext=(opt_x+1.5, total[np.argmin(total)]+1), fontsize=8, color='#43A047',
             arrowprops=dict(arrowstyle='->', color='#43A047'))

ax2.set_xlabel('Model Complexity'); ax2.set_ylabel('Error')
ax2.set_title('Bias-Variance Tradeoff', fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.suptitle('Decision Tree Overfitting & Bias-Variance Tradeoff', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bias_variance.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Optimal DT depth: {opt_depth}')

### Q2 · Coding: `plot_overfitting_curve()`

In [ ]:
def plot_overfitting_curve(X, y, max_depths, test_size=0.2, seed=SEED):
    """
    Train Decision Trees at each max_depth.
    Plot train vs test accuracy and return the optimal depth.

    Parameters
    ----------
    X          : feature DataFrame
    y          : target Series
    max_depths : iterable of int depths to test
    test_size  : fraction for test split (default 0.2)
    seed       : random state

    Returns
    -------
    optimal_depth : int
    """
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y)

    train_acc, test_acc = [], []

    for d in max_depths:
        tree = DecisionTreeClassifier(max_depth=d, random_state=seed)
        tree.fit(X_tr, y_tr)
        train_acc.append(accuracy_score(y_tr, tree.predict(X_tr)))
        test_acc.append(accuracy_score(y_te, tree.predict(X_te)))

    optimal_depth = list(max_depths)[int(np.argmax(test_acc))]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(max_depths, train_acc, 'b-o', label='Train accuracy', markersize=6)
    ax.plot(max_depths, test_acc,  'r-o', label='Test accuracy',  markersize=6)
    ax.axvline(optimal_depth, ls='--', color='green', lw=2,
               label=f'Optimal depth = {optimal_depth}')
    ax.fill_between(max_depths, train_acc, test_acc, alpha=0.1, color='red',
                    label='Overfit gap')
    ax.set_xlabel('max_depth'); ax.set_ylabel('Accuracy')
    ax.set_title('Decision Tree: Overfitting Curve', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('overfitting_curve.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f'Optimal max_depth = {optimal_depth}  '
          f'(test acc = {max(test_acc):.4f})')
    return optimal_depth


optimal = plot_overfitting_curve(X, y, max_depths=range(1, 20))

### Q3 · Debug: RF with identical train and test accuracy (0.95) — is it a problem?

In [ ]:
# Reproduce the scenario
rf_debug = RandomForestClassifier(
    n_estimators=500, max_depth=3, random_state=42)
rf_debug.fit(X_train, y_train)

train_score = rf_debug.score(X_train, y_train)
test_score  = rf_debug.score(X_test,  y_test)
print(f'Train: {train_score:.2f}   Test: {test_score:.2f}')

debug_answer = """
DEBUG ANSWER
============
Identical train ≈ test accuracy (0.95) is NOT necessarily a problem.
It is the IDEAL scenario — the model generalises well and is NOT overfitting.

Why it's fine:
  • max_depth=3 is a shallow setting that strongly regularises each tree,
    preventing them from memorising the training set.
  • Bagging adds further variance reduction, smoothing out any residual
    tree-level overfitting.
  • Equal scores mean the bias-variance tradeoff is well-balanced at 0.95.

When to investigate further:
  • If 0.95 is suspiciously high (near-perfect on a real noisy dataset)
    → check for data leakage (target-correlated features in X).
  • If the model seems too simple (high bias / underfit), try removing
    max_depth constraint and retrain.
  • Always cross-validate; a single train/test split can be coincidentally
    equal without reflecting true generalisability.

Verdict: Identical scores at 0.95 → healthy, well-generalising model. ✓
"""
print(debug_answer)

---
## Part D — AI-Augmented Task (10%): Comparison Infographic

In [ ]:
# Visual comparison: DT vs RF vs Logistic Regression for non-technical audience
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#F5F7FA')

models_info = [
    {
        'name'             : 'Logistic Regression',
        'icon'             : '📈',
        'color'            : '#1565C0',
        'light'            : '#E3F2FD',
        'interpretability' : 8,
        'accuracy'         : 6,
        'speed'            : 9,
        'handles_nonlinear': 3,
        'pros'             : ['Very fast to train', 'Easy to interpret', 'Works with few features',
                              'Outputs calibrated probabilities'],
        'cons'             : ['Assumes linear boundary', 'Needs feature engineering',
                              'Sensitive to outliers'],
        'use_when'         : 'You need a fast baseline and relationships are approximately linear'
    },
    {
        'name'             : 'Decision Tree',
        'icon'             : '🌳',
        'color'            : '#2E7D32',
        'light'            : '#E8F5E9',
        'interpretability' : 9,
        'accuracy'         : 6,
        'speed'            : 8,
        'handles_nonlinear': 8,
        'pros'             : ['Human-readable rules', 'No feature scaling needed',
                              'Handles mixed data types', 'Great for compliance'],
        'cons'             : ['Overfits without pruning', 'Unstable (high variance)',
                              'Biased toward high-cardinality features'],
        'use_when'         : 'Regulators or stakeholders need to read & verify the model logic'
    },
    {
        'name'             : 'Random Forest',
        'icon'             : '🌲🌲🌲',
        'color'            : '#E65100',
        'light'            : '#FFF3E0',
        'interpretability' : 4,
        'accuracy'         : 9,
        'speed'            : 6,
        'handles_nonlinear': 9,
        'pros'             : ['High accuracy out-of-the-box', 'Robust to outliers & noise',
                              'Built-in feature importance', 'Parallelisable'],
        'cons'             : ['Less interpretable', 'Slower than single tree',
                              'Large memory footprint'],
        'use_when'         : 'Accuracy is the priority and you have structured/tabular data'
    },
]

n_cols = 3
for col, info in enumerate(models_info):
    ax = fig.add_subplot(2, n_cols, col + 1)
    ax.set_facecolor(info['light'])
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')

    # Title
    ax.text(0.5, 0.93, info['icon'], ha='center', va='top', fontsize=22)
    ax.text(0.5, 0.80, info['name'], ha='center', va='top',
            fontsize=13, fontweight='bold', color=info['color'])

    # Pros
    ax.text(0.05, 0.68, '✅ Pros', fontsize=9, fontweight='bold', color='#2E7D32')
    for i, p in enumerate(info['pros'][:3]):
        ax.text(0.08, 0.61 - i*0.09, f'• {p}', fontsize=8, color='#333')

    # Cons
    ax.text(0.05, 0.33, '❌ Cons', fontsize=9, fontweight='bold', color='#B71C1C')
    for i, c in enumerate(info['cons'][:3]):
        ax.text(0.08, 0.26 - i*0.09, f'• {c}', fontsize=8, color='#333')

    # When to use
    ax.text(0.05, 0.04, f'🎯 Use when: {info["use_when"]}',
            fontsize=7.5, color=info['color'], wrap=True,
            style='italic')

# Bottom row: radar-style bar chart
ax_bar = fig.add_subplot(2, 1, 2)
ax_bar.set_facecolor('#FAFAFA')
categories  = ['Interpretability', 'Accuracy', 'Training Speed', 'Handles Non-linearity']
keys        = ['interpretability', 'accuracy', 'speed', 'handles_nonlinear']
x           = np.arange(len(categories))
width       = 0.25

for i, info in enumerate(models_info):
    vals = [info[k] for k in keys]
    bars = ax_bar.bar(x + i*width, vals, width,
                      label=info['name'], color=info['color'],
                      alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    str(v), ha='center', fontsize=8, fontweight='bold')

ax_bar.set_xticks(x + width)
ax_bar.set_xticklabels(categories, fontsize=11)
ax_bar.set_ylim(0, 12)
ax_bar.set_ylabel('Score (1–10)', fontsize=11)
ax_bar.legend(fontsize=10)
ax_bar.set_title('Model Comparison: Score on Key Dimensions', fontsize=12, fontweight='bold')
ax_bar.grid(axis='y', alpha=0.3)

fig.suptitle('Which ML Model Should You Use?\nLogistic Regression vs Decision Tree vs Random Forest',
             fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('model_comparison_infographic.png', dpi=120, bbox_inches='tight')
plt.show()
print('Infographic saved as model_comparison_infographic.png')

In [ ]:
# AI critique
critique = """
INFOGRAPHIC EVALUATION & CRITIQUE
===================================
Accurate aspects:
  ✓ LR interpretability is correctly scored highest for LINEAR scenarios.
  ✓ DT is marked as best for compliance/regulatory use.
  ✓ RF accuracy advantage over single DT is well-represented.
  ✓ Speed ordering (LR > DT > RF) is generally correct.

Potential oversimplifications:
  ! 'Interpretability' scores are context-dependent. A shallow DT (depth≤4)
    IS highly interpretable, but a deep DT (depth=20) is not.
  ! LR IS interpretable but only for linear relationships; this nuance
    is lost in a single score.
  ! RF accuracy=9 may be dataset-specific; on very high-dimensional data
    or with small n, LR can match RF.

Corrections applied:
  + Added 'Handles Non-linearity' as a fourth dimension (AI-generated
    version only showed 3).
  + Added 'Use when' section to guide non-technical readers.
  + Separated pros/cons per model rather than generic statements.
"""
print(critique)

---
## Summary Table

In [ ]:
print('FINAL MODEL COMPARISON')
print('=' * 70)
print(results.round(4).to_string())
print('\nFiles generated:')
for f in ['dt_tree.png','confusion_matrices.png','feature_importance.png',
          'rf_vs_et.png','bias_variance.png','overfitting_curve.png',
          'model_comparison_infographic.png']:
    print(f'  • {f}')